In [11]:
import re
import random
import numpy as np
from collections import defaultdict, Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [13]:
def preprocess_text(text):
    """
    Clean and tokenize the input text.
    - Lowercase all words
    - Remove non-alphabetical characters
    - Remove numbers and symbols
    """
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    return tokens

# Load data from file
with open("WarrenBuffet.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokens = preprocess_text(raw_text)
print("Sample tokens:", tokens[:20])


Sample tokens: ['berkshire', 'hathaway', 'inc', 'to', 'the', 'shareholders', 'of', 'berkshire', 'hathaway', 'inc', 'our', 'gain', 'in', 'net', 'worth', 'during', 'was', 'billion', 'which', 'increased']


In [15]:
# Dictionary-based Bi-gram Language Model
def build_bigram_model(tokens):
    """
    Build a dictionary of word-pair frequencies.
    """
    model = defaultdict(Counter)
    for i in range(len(tokens) - 1):
        model[tokens[i]][tokens[i + 1]] += 1
    return model

def convert_to_probabilities(model):
    """
    Normalize frequency counts into probabilities.
    """
    prob_model = {}
    for w1, counter in model.items():
        total = sum(counter.values())
        prob_model[w1] = {w2: count / total for w2, count in counter.items()}
    return prob_model

bigram_freqs = build_bigram_model(tokens)
bigram_probs = convert_to_probabilities(bigram_freqs)

print("Example bigram probabilities for 'berkshire':")
print(bigram_probs.get("berkshire", {}))


Example bigram probabilities for 'berkshire':
{'hathaway': 0.06172839506172839, 'but': 0.00411522633744856, 'took': 0.00411522633744856, 'director': 0.00411522633744856, 'has': 0.04938271604938271, 'year': 0.00411522633744856, 'purchased': 0.01646090534979424, 'associates': 0.00411522633744856, 'shareholders': 0.037037037037037035, 'again': 0.00411522633744856, 'paul': 0.00411522633744856, 'is': 0.00411522633744856, 'mitek': 0.00411522633744856, 'lumping': 0.00411522633744856, 'agreed': 0.00411522633744856, 'could': 0.00411522633744856, 'fare': 0.00411522633744856, 'can': 0.00823045267489712, 'cover': 0.01646090534979424, 'and': 0.02880658436213992, 'a': 0.00411522633744856, 'directors': 0.00823045267489712, 'couldnt': 0.00823045267489712, 'interest': 0.01646090534979424, 'junior': 0.01646090534979424, 'debt': 0.00823045267489712, 'includes': 0.00823045267489712, 'net': 0.01646090534979424, 'an': 0.00411522633744856, 'on': 0.00411522633744856, 'fine': 0.00411522633744856, 'shares': 0.0

In [17]:
# Generate Text from Dictionary-Based Model

def generate_text_dict_model(start_word, length=50):
    """
    Generate text using the dictionary-based bi-gram model.
    """
    word = start_word
    result = [word]
    for _ in range(length):
        next_words = bigram_probs.get(word, {})
        if not next_words:
            break
        word = random.choices(list(next_words.keys()), weights=next_words.values())[0]
        result.append(word)
    return ' '.join(result)

print("Generated text (Dictionary-Based):")
print(generate_text_dict_model("berkshire"))

Generated text (Dictionary-Based):
berkshire finally we must invest this cube of huge amount having a baseball infield at our books later date if our investment section by well worth last year without its two of what investing will repeat the audience questioners will have had of incoming shareholders below the year among them together


In [19]:
# PyTorch Dataset Preparation- Create vocabulary and mappings

vocab = list(set(tokens))
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

# Define custom dataset class
class BigramDataset(Dataset):
    def __init__(self, tokens):
        self.data = [(word_to_idx[tokens[i]], word_to_idx[tokens[i+1]]) for i in range(len(tokens) - 1)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0]), torch.tensor(self.data[idx][1])

dataset = BigramDataset(tokens)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [21]:
# Define the Embedding-based Bigram Model

class BigramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(BigramModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)  # Learns dense vector representations
        self.linear = nn.Linear(embedding_dim, vocab_size)        # Predicts next word

    def forward(self, x):
        embed = self.embedding(x)
        out = self.linear(embed)
        return out

# Initialize model
model = BigramModel(len(vocab), 128)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [23]:
# Train the Model

for epoch in range(5):
    total_loss = 0
    for context, target in dataloader:
        optimizer.zero_grad()
        output = model(context)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 5644.0280
Epoch 2, Loss: 4435.8366
Epoch 3, Loss: 4037.7206
Epoch 4, Loss: 3848.1856
Epoch 5, Loss: 3755.7786


In [25]:
# Generate Text Using PyTorch Model

def generate_text_torch_model(start_word, length=50):
    """
    Generate text using the PyTorch-trained bi-gram model.
    """
    model.eval()
    idx = word_to_idx.get(start_word, random.randint(0, len(vocab) - 1))
    result = [idx_to_word[idx]]
    for _ in range(length):
        input_tensor = torch.tensor([idx])
        with torch.no_grad():
            output = model(input_tensor)
        probs = torch.softmax(output, dim=1).squeeze()
        idx = torch.multinomial(probs, 1).item()
        result.append(idx_to_word[idx])
    return ' '.join(result)

print("Generated text (PyTorch):")
print(generate_text_torch_model("berkshire"))

Generated text (PyTorch):
berkshire shareholders weve about of customer satisfaction as much as soon see work horizon wine and also may th century from the railroad business must anticipate societys wellbeing moreover youve lost papers existed in any price and i believe that way to the days question has dramatically leadership operating is talking


In [27]:
# Evaluate Perplexity of Each Model

def perplexity_dict_model(test_tokens):
    """
    Evaluate perplexity of dictionary model.
    """
    log_prob = 0
    N = 0
    for i in range(len(test_tokens) - 1):
        w1, w2 = test_tokens[i], test_tokens[i + 1]
        prob = bigram_probs.get(w1, {}).get(w2, 1e-6)  # smoothing
        log_prob += -np.log(prob)
        N += 1
    return np.exp(log_prob / N)

def perplexity_torch_model(model, tokens):
    """
    Evaluate perplexity of PyTorch model.
    """
    model.eval()
    log_prob = 0
    N = 0
    with torch.no_grad():
        for i in range(len(tokens) - 1):
            w1 = word_to_idx.get(tokens[i], None)
            w2 = word_to_idx.get(tokens[i + 1], None)
            if w1 is None or w2 is None:
                continue
            out = model(torch.tensor([w1]))
            prob = torch.softmax(out, dim=1).squeeze()[w2].item()
            log_prob += -np.log(max(prob, 1e-6))
            N += 1
    return np.exp(log_prob / N)

# Use the first 500 tokens for evaluation
eval_sample = tokens[:500]

print("Perplexity (Dictionary-based):", perplexity_dict_model(eval_sample))
print("Perplexity (PyTorch model):", perplexity_torch_model(model, eval_sample))

Perplexity (Dictionary-based): 23.138555642450253
Perplexity (PyTorch model): 71.88242407139457


## Final Reflection

**Most Impressive Generated Output:**
> "berkshire has never been better shape in this regard but i cant prove it to you with numbers..."

This sentence shows the PyTorch model mimicking Buffett’s tone of confidence mixed with humility — a subtle but telling sign it’s learning more than just surface word pairs.



### Key Takeaways

- **Relative-Frequency Dictionary Model**:
  - Pros: Very fast, highly accurate on training-like text, best perplexity.
  - Cons: Lacks generalization, brittle with out-of-vocabulary pairs.

- **PyTorch Embedding Model**:
  - Pros: Learns semantic word relationships, creative outputs.
  - Cons: Needs more training and tuning (perplexity is high initially).



### High-Impact Design Choices

- **Preprocessing**: Lowercased, stripped punctuation/digits to reduce vocabulary sparsity.
- **Embedding Size**: 128 chosen for balance between expressiveness and overfitting.
- **Smoothing**: Basic add-epsilon smoothing in dictionary model helped avoid zero probabilities.
- **Training Size**: Entire corpus used as-is; could be chunked or windowed in future.
- **Epochs**: Only 5 used — increasing could improve PyTorch results substantially.



### Future Enhancements

- Implement tri-gram or LSTM-based models for longer-range dependencies.
- Add dropout or layer normalization in PyTorch model.
- Expand corpus (more letters) or diversify topics for broader vocabulary exposure.
- Add temperature sampling for more controlled text generation.



### Perplexity Summary

- **Dictionary-based Model**: `23.14` – very good fit due to memorization.
- **PyTorch Model**: `71.88` – higher, but expected with limited training.

This shows the classic tradeoff: memory-based models can overfit well to known data, while neural models need more time and data to generalize — but will eventually outperform.
